# ETHICS Gender — revisi gambar Bab VI (8 Oktober 2026)

Notebook presentasi mandiri: jalankan semua sel dari root repo atau folder notebook. Tidak menjalankan notebook lama, mengubah PNG aktif, atau menimpa hasil. Setiap eksekusi membuat run baru. Enam jenis gambar menjadi sembilan PNG/PDF karena heatmap dipecah. Man/Woman dipertahankan.

## SPEC dan DESIGN PLAN

Lihat `docs/ethics_gender_figures_bab6_revision_20261008.md`. Seluruh tabel sumber dipakai tanpa sampling. Akurasi gabungan Commonsense + Justice dipertahankan sebagai ringkasan deskriptif sesuai permintaan; tabel per subset dan coverage tetap tersedia. Tidak ada pengujian ulang atau perubahan parser.


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib, json, textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import Normalize

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks/gender/ethics accuracy analysis/tables").is_dir())
ACTIVE = ROOT / "notebooks/gender/ethics accuracy analysis"
TABLES = ACTIVE / "tables"
RUN = ROOT / "result/benchmarks/ethics/reports" / ("gender_ethics_bab6_revision_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
RUN.mkdir(parents=True, exist_ok=False)
FIG = RUN / "bab6"
FIG.mkdir()
SUP = RUN / "tables"
SUP.mkdir()
PROOF = RUN / "proof"
PROOF.mkdir()
def digest(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()
protected = list(TABLES.glob("*.csv")) + list((ACTIVE / "figures").glob("*")) + list(ACTIVE.glob("*.ipynb"))
baseline = {str(p.relative_to(ROOT)): digest(p) for p in protected if p.is_file()}
def read(name):
    return pd.read_csv(TABLES / (name + ".csv"))
accuracy = read("ethics_accuracy_by_model_persona_category")
summary = read("ethics_condition_summary_source_copy")
status = read("status_by_model_persona")
reasons = read("abstain_reason_by_model_persona")
combined = read("ethics_combined_four_subsets")
inference = read("ethics_accuracy_inference_construct_valid")
models = combined.model.drop_duplicates().tolist()
model_names = dict(combined[["model", "model_label"]].drop_duplicates().values)
short_names = ["Qwen3 Next\n80B-A3B", "Llama 3.1\n8B", "Ministral\n8B", "DeepSeek R1\nDistill 32B", "GPT-OSS\n20B", "Llama 3.3\n70B"]
row_names = ["Qwen3 Next 80B-A3B", "Llama 3.1 8B", "Ministral 8B", "DeepSeek R1-Distill 32B", "GPT-OSS 20B", "Llama 3.3 70B"]
personas = ["man_explicit_1", "man_implicit_1", "man_implicit_2", "man_implicit_3", "woman_explicit_1", "woman_implicit_1", "woman_implicit_2", "woman_implicit_3", "neutral"]
persona_labels = ["Man E", "Man I1", "Man I2", "Man I3", "Woman E", "Woman I1", "Woman I2", "Woman I3", "N"]
categories = ["Man E", "Man I", "Woman E", "Woman I", "Neutral"]
colors = ["#0047AB", "#0086A8", "#B80F2E", "#E45700", "#30343B"]
plt.rcParams.update({"font.family":"DejaVu Sans", "font.size":8, "axes.titlesize":9, "axes.labelsize":8, "xtick.labelsize":7, "ytick.labelsize":7, "legend.fontsize":7, "pdf.fonttype":42, "savefig.facecolor":"white"})
checks = {}
captions = {}
artifacts = []
cell_audit = []
assert len(models) == 6 and len(inference) == 216
assert summary.total.sum() == 81756
assert status.abstain.sum() == 18521
assert not inference.duplicated(["subset", "model", "display_order"]).any()
for table in [status, combined, accuracy, summary]:
    assert (table.total == table.committed + table.abstain + table.missing).all()
    np.testing.assert_allclose(table.commit_rate, table.committed / table.total)
np.testing.assert_allclose(accuracy.accuracy_committed, accuracy.n_correct / accuracy.committed)
np.testing.assert_allclose(combined.committed, combined.n_wrong + combined.n_acceptable)
np.testing.assert_allclose(reasons.pct_of_abstain, reasons.reason_count / reasons.abstain)
reason_totals = reasons.groupby(["model","persona"]).reason_count.sum()
pd.testing.assert_series_equal(reason_totals.sort_index(), status.set_index(["model","persona"]).abstain.sort_index(), check_names=False)
checks["exclusive_reason_counts_sum_to_abstain"] = True
# Verify the requested descriptive pooling against per-subset source counts.
valid = summary[summary.subset.isin(["commonsense", "justice"])].copy()
def category_for(persona):
    if persona == "neutral": return "Neutral"
    return ("Man" if persona.startswith("man_") else "Woman") + (" E" if "_explicit_" in persona else " I")
valid["category_short"] = valid.persona.map(category_for)
count_cols = ["total","committed","abstain","missing","n_correct"]
pooled = valid.groupby(["model","category_short"])[count_cols].sum().sort_index()
pd.testing.assert_frame_equal(pooled, accuracy.set_index(["model","category_short"])[count_cols].sort_index(), check_dtype=False)
checks["pooled_counts_match_per_subset"] = True
coverage_cols = ["model","model_label","category_short","total","committed","commit_rate","abstain","abstain_rate","missing","missing_rate","n_correct","accuracy_committed"]
accuracy[coverage_cols].to_csv(SUP / "accuracy_with_coverage.csv", index=False)
valid.drop(columns="category_short").to_csv(SUP / "accuracy_per_subset_with_coverage.csv", index=False)
status.to_csv(SUP / "status_and_abstain_denominators.csv", index=False)
inference.to_csv(SUP / "effect_inference_source_copy.csv", index=False)
summary[summary.subset.isin(["deontology","virtue"])].drop(columns=["n_correct","accuracy_committed","accuracy_committed_excl_conflict","effective_success_rate_nonmissing"], errors="ignore").to_csv(SUP / "stance_with_coverage.csv", index=False)

def dec(value, digits=1, signed=False):
    return (f"{value:+.{digits}f}" if signed else f"{value:.{digits}f}").replace(".", ",").replace("-", "−")

def save(fig, stem, caption):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for ax in fig.axes:
        for t in ax.texts:
            if getattr(t, "_cell_annotation", False):
                box = t.get_window_extent(renderer)
                x, y = t.get_position()
                p0 = ax.transData.transform((x-.5,y-.5))
                p1 = ax.transData.transform((x+.5,y+.5))
                fits = box.width < abs(p1[0]-p0[0]) and box.height < abs(p1[1]-p0[1])
                cell_audit.append({"figure":stem,"text":t.get_text(),"fits":bool(fits)})
                assert fits, (stem, t.get_text(), box.width, abs(p1[0]-p0[0]))
    for ext in ["png","pdf"]:
        path = FIG / (stem + "." + ext)
        assert not path.exists()
        fig.savefig(path, dpi=300)
        artifacts.append(path)
    captions[stem] = caption
    plt.close(fig)
    print("Selesai:", stem, flush=True)

def heat(ax, data, xlabels, ylabels, *, vmin=0, vmax=100, cmap="YlOrBr", annotations=None, fontsize=7):
    data = np.asarray(data, dtype=float)
    im = ax.pcolormesh(np.arange(data.shape[1]+1)-.5, np.arange(data.shape[0]+1)-.5, np.ma.masked_invalid(data), cmap=cmap, vmin=vmin, vmax=vmax, shading="flat", rasterized=False)
    ax.set_ylim(data.shape[0]-.5, -.5)
    ax.set_xlim(-.5, data.shape[1]-.5)
    ax.set_xticks(range(data.shape[1]), xlabels)
    ax.set_yticks(range(data.shape[0]), ylabels)
    ax.tick_params(length=0)
    ax.set_xticks(np.arange(-.5, data.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-.5, data.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=.5)
    ax.tick_params(which="minor", bottom=False, left=False)
    for spine in ax.spines.values(): spine.set_visible(False)
    for (i,j), value in np.ndenumerate(data):
        label = annotations[i,j] if annotations is not None else (dec(value) if np.isfinite(value) else "NA")
        rgba = im.cmap(im.norm(value)) if np.isfinite(value) else (1,1,1,1)
        luminance = .2126*rgba[0] + .7152*rgba[1] + .0722*rgba[2]
        t = ax.text(j,i,label,ha="center",va="center",fontsize=fontsize, color="white" if luminance<.5 else "#202020")
        t._cell_annotation = True
    return im
print("Run baru:", RUN, flush=True)


## IMPLEMENTATION — alasan abstain dan heatmap efek

Alasan abstain memakai kategori eksklusif yang sudah tersimpan, tidak melakukan klasifikasi teks baru. Simbol efek mempertahankan keputusan Holm; dagger hanya menandai prasyarat Q untuk antarskenario.


In [ ]:
reason_order = reasons.groupby("reason_label").reason_count.sum().sort_values(ascending=False).index.tolist()
reason_labels = reason_order
reason_export = []
for part in range(2):
    fig, axes = plt.subplots(3,1,figsize=(5.51,7.65))
    fig.subplots_adjust(left=.39,right=.98,bottom=.14,top=.935,hspace=.46)
    for ax, model in zip(axes, models[part*3:part*3+3]):
        frame = reasons[reasons.model.eq(model)]
        count = frame.pivot(index="reason_label",columns="persona",values="reason_count").reindex(index=reason_order,columns=personas).fillna(0)
        denom = status[status.model.eq(model)].set_index("persona").loc[personas,"abstain"]
        matrix = count.div(denom.replace(0,np.nan),axis=1)*100
        labels = [lab.replace(" ","\n") + "\nn=" + str(int(n)) for lab,n in zip(persona_labels,denom)]
        im = heat(ax,matrix,labels,reason_labels,fontsize=6.8)
        ax.tick_params(axis="x",labelsize=6.4,pad=3)
        ax.tick_params(axis="y",labelsize=6.5,pad=3)
        ax.set_title(model_names[model],pad=4,fontsize=8)
        fig.canvas.draw()
        tick_boxes = [t.get_window_extent(fig.canvas.get_renderer()) for t in ax.get_yticklabels()]
        assert all(not a.overlaps(b) for a,b in zip(tick_boxes,tick_boxes[1:]))
        assert all(b.x0 >= 0 for b in tick_boxes)
        for reason in reason_order:
            for persona in personas:
                reason_export.append({"model":model,"persona":persona,"reason_label":reason,"reason_count":int(count.loc[reason,persona]),"abstain":int(denom[persona]),"pct_of_abstain":matrix.loc[reason,persona]})
    cax=fig.add_axes([.38,.045,.54,.013])
    fig.colorbar(im,cax=cax,orientation="horizontal",ticks=[0,25,50,75,100],label="% dari abstain")
    fig.suptitle(f"ETHICS Gender: alasan abstain ({part+1}/2)",fontsize=10,y=.986)
    save(fig,f"04{chr(97+part)}_ethics_abstain_reason_share",
         f"Alasan abstain, bagian {part+1}/2. Persentase dari seluruh respons abstain pada model dan skenario terkait, digabung dari empat subset. n pada label kolom adalah denominator abstain; bukan seluruh respons. Kategori alasan bersifat eksklusif pada tabel sumber. Warna memakai rentang 0–100% yang sama untuk semua model. N = netral; E = eksplisit; I1–I3 adalah variasi isi warmup, bukan tingkat intensitas.")
pd.DataFrame(reason_export).to_csv(SUP / "abstain_reason_plot_values.csv",index=False)

holm = inference.holm_adjusted_p_value.lt(.05)
q = inference.omnibus_cochran_q_p_value.lt(.05)
cross = inference.identity.eq("man_vs_woman")
assert (holm == inference.significant_holm_0_05).all()
assert ((holm & (q | cross)) == inference.decision_significant_0_05).all()
checks["significance_flags_match_saved_pvalues"] = True
effect_limit = np.ceil(inference.effect_size.abs().max()*10)/10
specs = [
    ("06a_ethics_effect_persona_vs_neutral","Persona − N",[0,1,2,3,7,8,9,10],["Man\nE−N","Man\nI1−N","Man\nI2−N","Man\nI3−N","Woman\nE−N","Woman\nI1−N","Woman\nI2−N","Woman\nI3−N"]),
    ("06b_ethics_effect_implicit_vs_explicit","I − E",[4,5,6,11,12,13],["Man\nI1−E","Man\nI2−E","Man\nI3−E","Woman\nI1−E","Woman\nI2−E","Woman\nI3−E"]),
    ("06c_ethics_effect_man_vs_woman","Man − Woman",[14,15,16,17],["E","I1","I2","I3"]),
]
seen = []
for stem,title,orders,labels in specs:
    fig,axes=plt.subplots(2,1,figsize=(5.51,5.55))
    fig.subplots_adjust(left=.31,right=.98,bottom=.20,top=.90,hspace=.48)
    for ax,subset in zip(axes,["commonsense","justice"]):
        frame=inference[inference.subset.eq(subset) & inference.display_order.isin(orders)]
        matrix=frame.pivot(index="model",columns="display_order",values="effect_size").reindex(index=models,columns=orders)
        annotation=np.empty(matrix.shape,dtype=object)
        indexed=frame.set_index(["model","display_order"])
        for i,model in enumerate(models):
            for j,order in enumerate(orders):
                row=indexed.loc[(model,order)]
                marker=("*" if row.significant_holm_0_05 else "") + ("†" if row.significant_holm_0_05 and row.identity!="man_vs_woman" and row.omnibus_cochran_q_p_value<.05 else "")
                annotation[i,j]=dec(row.effect_size,2,True) + ("\n"+marker if marker else "")
                seen.append((subset,model,order))
        im=heat(ax,matrix,labels,row_names,vmin=-effect_limit,vmax=effect_limit,cmap="RdBu_r",annotations=annotation,fontsize=7.2)
        ax.set_title(subset.title(),fontsize=9,pad=5)
        ax.tick_params(axis="x",pad=4,labelsize=7)
        ax.tick_params(axis="y",labelsize=6.8)
    fig.suptitle("ETHICS Gender: ukuran efek akurasi\n"+title,fontsize=10,y=.985)
    cax=fig.add_axes([.34,.105,.60,.022])
    fig.colorbar(im,cax=cax,orientation="horizontal",label="Cohen’s h",ticks=np.linspace(-effect_limit,effect_limit,5))
    note="* Holm < 0,05; † juga Q Cochran < 0,05." if orders[0]<14 else "* Holm < 0,05; Man−Woman tidak memakai prasyarat Q."
    fig.text(.5,.016,note,ha="center",fontsize=7)
    save(fig,stem,
         "Ukuran efek Cohen’s h pada akurasi pasangan respons yang sama-sama committed, dipisahkan menurut Commonsense dan Justice. Positif berarti akurasi sisi kiri lebih tinggi. Semua model dan kontras ditampilkan; skala warna sama pada tiga gambar efek. * menandakan p Exact McNemar setelah koreksi Holm < 0,05 (bukan tingkatan jumlah bintang). Untuk antarskenario, † tambahan menandakan p Q Cochran < 0,05; hasil utama mensyaratkan keduanya. Q menggunakan himpunan lengkap lima skenario dan dapat berbeda dari himpunan pasangan McNemar. Man−Woman tidak memakai prasyarat Q. Keluarga koreksi tidak berubah: tujuh kontras antarskenario per model/subset/identitas, empat kontras Man−Woman per model/subset. N = netral; I1–I3 tetap variasi isi warmup.")
assert len(seen)==len(set(seen))==216
checks["all_216_effect_cells_preserved"] = True


## IMPLEMENTATION — bar dan distribusi keputusan


In [ ]:
# Keep all numerical pooling unchanged; only units and labels change.
pivot=accuracy.pivot(index="model",columns="category_short",values="accuracy_committed").reindex(index=models,columns=categories)
coverage=accuracy.pivot(index="model",columns="category_short",values="commit_rate").reindex(index=models,columns=categories)
fig,(ax,covax)=plt.subplots(2,1,figsize=(5.51,5.25),gridspec_kw={"height_ratios":[3.6,1]})
fig.subplots_adjust(left=.13,right=.98,top=.84,bottom=.09,hspace=.50)
x=np.arange(6)
for j,category in enumerate(categories):
    bars=ax.bar(x+(j-2)*.16,pivot[category]*100,width=.155,color=colors[j],label=category)
    ax.bar_label(bars,labels=[dec(v,1) for v in pivot[category]*100],padding=2,rotation=90,fontsize=6.8)
ax.set_ylim(0,102)
ax.set_xticks(x,short_names,fontsize=6.7)
ax.set_ylabel("Akurasi pada committed (%)")
ax.set_axisbelow(True)
ax.grid(axis="y",alpha=.2)
ax.legend(ncol=3,loc="lower center",bbox_to_anchor=(.5,1.01),frameon=False)
fig.suptitle("ETHICS Gender: akurasi Commonsense + Justice\nRingkasan deskriptif pada respons committed",fontsize=10,y=.98)
heat(covax,coverage.to_numpy().T*100,short_names,categories,cmap="Greys",fontsize=6.8)
covax.set_title("Coverage pendamping: commit rate (%)",fontsize=8)
covax.tick_params(axis="x",labelsize=6.5)
save(fig,"01_ethics_absolute_accuracy_five_categories",
     "Ringkasan deskriptif akurasi Commonsense + Justice: jumlah benar dibagi jumlah committed. Abstain dan missing tidak masuk penyebut akurasi. I menggabungkan hitungan I1–I3, bukan rata-rata sederhana persentase. Netral adalah pembanding masing-masing model; tidak ada Neutral Avg antarmodel. Panel bawah menunjukkan commit rate dengan penyebut seluruh respons dalam kategori. Perbedaan coverage dapat mengubah komposisi respons yang dinilai. Tabel accuracy_with_coverage.csv memuat total, committed, abstain, missing beserta rate; hasil per subset tersedia dalam accuracy_per_subset_with_coverage.csv. Agregat ini tidak menggantikan analisis per subset.")
delta=pd.DataFrame({"Man I−E":100*(pivot["Man I"]-pivot["Man E"]), "Woman I−E":100*(pivot["Woman I"]-pivot["Woman E"]), "Man I−Woman I":100*(pivot["Man I"]-pivot["Woman I"])})
delta.to_csv(SUP / "descriptive_accuracy_delta_pp.csv")
fig,axes=plt.subplots(2,1,figsize=(5.51,5.45))
fig.subplots_adjust(left=.31,right=.97,bottom=.10,top=.87,hspace=.50)
limit=np.ceil(np.abs(delta.to_numpy()).max())+2
for ax,columns,title,palette in [
    (axes[0],["Man I−E","Woman I−E"],"I − E",colors[::2][:2]),
    (axes[1],["Man I−Woman I"],"Man I − Woman I",["#5B21B6"])]:
    y=np.arange(6)
    for j,(column,color) in enumerate(zip(columns,palette)):
        offset=(j-(len(columns)-1)/2)*.34
        bars=ax.barh(y+offset,delta[column],height=.31 if len(columns)>1 else .5,label=column,color=color)
        ax.bar_label(bars,labels=[dec(v,2,True) for v in delta[column]],padding=3,fontsize=7)
    ax.axvline(0,color="#30343B",lw=.8)
    ax.set_xlim(-limit,limit)
    ax.set_yticks(y,row_names,fontsize=6.8)
    ax.invert_yaxis()
    ax.set_xlabel("Selisih akurasi (poin persentase)")
    ax.set_title(title,fontsize=9)
    ax.grid(axis="x",alpha=.15)
    ax.set_axisbelow(True)
    if len(columns)>1: ax.legend(ncol=2,loc="lower center",bbox_to_anchor=(.5,1.09),frameon=False)
fig.suptitle("ETHICS Gender: delta akurasi\nCommonsense + Justice — ringkasan deskriptif",fontsize=10,y=.985)
save(fig,"03_ethics_delta_implicit_vs_explicit",
     "Selisih akurasi dalam poin persentase untuk Commonsense + Justice. Panel atas: Man I−E dan Woman I−E. Panel bawah: Man I−Woman I (kategori implicit saja, sesuai perhitungan batang ungu lama). I menggabungkan hitungan I1–I3. Delta dihitung dari akurasi marginal setiap kategori pada respons committed, bukan otomatis dari soal yang sama-sama committed pada kedua kondisi. Tidak ada klaim signifikansi untuk batang agregat; pengujian berpasangan tetap per skenario/subset. Baca coverage pada gambar 01 dan tabel accuracy_with_coverage.csv.")

fig,axes=plt.subplots(3,2,figsize=(5.51,6.6))
fig.subplots_adjust(left=.10,right=.98,bottom=.13,top=.84,hspace=.65,wspace=.27)
status_colors=["#B80F2E","#0086A8","#E9C46A","#EE6C4D"]
for ax,model in zip(axes.flat,models):
    frame=combined[combined.model.eq(model)].set_index("category_short").loc[categories]
    bottom=np.zeros(5)
    for column,label,color in zip(["n_wrong","n_acceptable","abstain","missing"],["Wrong","Acceptable","Abstain","Missing"],status_colors):
        values=frame[column]/frame.total*100
        ax.bar(np.arange(5),values,bottom=bottom,color=color,label=label,width=.8)
        for j,v in enumerate(values):
            if v>=8: ax.text(j,bottom[j]+v/2,dec(v),ha="center",va="center",fontsize=6.8,color="white" if label in ["Wrong","Acceptable"] else "#202020")
        bottom+=values.to_numpy()
    np.testing.assert_allclose(bottom,100)
    ax.set_ylim(0,100)
    ax.set_yticks([0,50,100])
    ax.set_xticks(range(5),["Man\nE","Man\nI","Woman\nE","Woman\nI","N"],fontsize=6.8)
    ax.set_title(model_names[model].replace("DeepSeek R1-Distill","DeepSeek R1"),fontsize=8)
    ax.set_ylabel("% seluruh respons",fontsize=7)
fig.legend(*axes[0,0].get_legend_handles_labels(),loc="upper center",bbox_to_anchor=(.5,.918),ncol=4,frameon=False)
fig.suptitle("ETHICS Gender: distribusi jawaban\nEmpat subset",fontsize=10,y=.987)
fig.text(.5,.033,"I = gabungan hitungan I1–I3; N = netral.\nWrong = keputusan tindakan salah, bukan jawaban model salah.",ha="center",fontsize=7)
save(fig,"00_ethics_combined_four_subsets_distribution",
     "Distribusi keputusan Wrong/Acceptable serta abstain/missing dari keempat subset ETHICS. Penyebut setiap batang adalah seluruh respons, bukan hanya committed; jumlah empat bagian 100%. Wrong berarti model menilai tindakan salah, bukan jawaban model yang salah terhadap ground truth. I menggabungkan hitungan I1–I3, bukan rata-rata sederhana persentase. Subset dengan lebih banyak respons berbobot lebih besar. E = eksplisit; N = netral. Tidak ada uji akurasi gabungan empat subset.")

fig,axes=plt.subplots(2,1,figsize=(5.51,5.6))
fig.subplots_adjust(left=.31,right=.98,bottom=.20,top=.89,hspace=.48)
for ax,subset in zip(axes,["deontology","virtue"]):
    matrix=summary[summary.subset.eq(subset)].pivot(index="model",columns="persona",values="pct_answer_wrong").reindex(index=models,columns=personas)*100
    im=heat(ax,matrix,[s.replace(" ","\n") for s in persona_labels],row_names,cmap="RdYlBu_r",fontsize=7)
    ax.set_title(subset.title(),fontsize=9)
    ax.tick_params(axis="y",labelsize=6.8)
fig.suptitle("ETHICS Gender: distribusi keputusan\nDeontology dan Virtue",fontsize=10,y=.984)
cax=fig.add_axes([.34,.085,.60,.022])
fig.colorbar(im,cax=cax,orientation="horizontal",label="Wrong (% dari committed)",ticks=[0,25,50,75,100])
save(fig,"08_ethics_stance_wrong_construct_invalid_heatmap",
     "Persentase keputusan Wrong di antara respons committed pada Deontology dan Virtue; bukan akurasi terhadap label acuan. Wrong berarti model menilai tindakan salah. Kedua subset dan I1–I3 ditampilkan terpisah. Warna 0–100% memakai denominator committed per model/skenario/subset. Coverage tersedia pada tabel stance_with_coverage.csv. I1–I3 merupakan variasi isi warmup, bukan urutan tingkat implisit.")


## CHECKLIST, TESTING, dan INTEGRATION

Sel terakhir memvalidasi artefak dan hash. PDF setiap gambar dibuat pada lebar 14 cm (5,51 inci). Proof A4 berikut mempertahankan ukuran gambar tersebut, bukan mengandalkan DPI PNG. Cetak fisik dan pemasangan `.tex` tidak dilakukan.


In [ ]:
assert len(captions)==9
assert len(artifacts)==18 and all(p.stat().st_size>0 for p in artifacts)
assert all(digest(ROOT/p)==h for p,h in baseline.items())
checks["protected_source_files_unchanged"] = True
checks["all_annotated_heatmap_cells_fit"] = all(row["fits"] for row in cell_audit)
checks["nine_png_and_nine_pdf"] = True
checks["effect_shared_color_limits"] = [-float(effect_limit),float(effect_limit)]
checks["abstain_shared_color_limits"] = [0,100]
checks["holm_significant_cells"] = int(holm.sum())
checks["antarscenario_holm_and_q_cells"] = int((holm&q&~cross).sum())
checks["cross_identity_holm_cells"] = int((holm&cross).sum())
checks["antarscenario_holm_only_cells"] = int((holm&~q&~cross).sum())
pd.DataFrame(cell_audit).to_csv(SUP / "heatmap_cell_layout_audit.csv",index=False)
# The PNG proof is a convenient print-sized preview; individual figure PDFs are vector.
with PdfPages(PROOF / "A4_gambar_padat_14cm.pdf") as pages:
    for stem in ["04a_ethics_abstain_reason_share","04b_ethics_abstain_reason_share","06a_ethics_effect_persona_vs_neutral"]:
        fig=plt.figure(figsize=(8.2677,11.6929))
        image=plt.imread(FIG/(stem+".png"))
        height=5.51*image.shape[0]/image.shape[1]
        ax=fig.add_axes([.17,1-(1.0+height)/11.6929,5.51/8.2677,height/11.6929])
        ax.imshow(image)
        ax.axis("off")
        fig.text(.17,.956,"Proof A4 — lebar gambar 14 cm; cetak 100%",fontsize=10)
        y=1-(1.12+height)/11.6929
        brief=" ".join(captions[stem].split()[:65])+"…"
        fig.text(.17,y,textwrap.fill(brief,92),fontsize=9,va="top")
        pages.savefig(fig)
        plt.close(fig)
caption_text="# Caption revisi ETHICS gender\n\n"+"\n\n".join("## "+s+"\n\n"+c for s,c in captions.items())+"\n"
(RUN/"CAPTIONS.md").write_text(caption_text,encoding="utf-8")
guide="# Hasil revisi gambar ETHICS gender\n\nFolder bab6 berisi 9 PNG dan 9 PDF: enam jenis gambar, dengan heatmap efek menjadi tiga gambar dan alasan abstain menjadi dua gambar.\n\n"+"\n".join("- ["+s+".png](bab6/"+s+".png)" for s in captions)+"\n\nCaption lengkap: CAPTIONS.md. Tabel coverage dan nilai plot: tables/. Proof halaman padat: proof/A4_gambar_padat_14cm.pdf. PDF setiap gambar adalah vektor, lebar 14 cm.\n\nTidak mengubah gambar lama, data, parser, pengujian, atau naskah TA. Pemeriksaan ini bukan validasi ulang statistik atau cetak fisik.\n"
(RUN/"MULAI_DI_SINI.md").write_text(guide,encoding="utf-8")
(RUN/"checks.json").write_text(json.dumps(checks,indent=2),encoding="utf-8")
manifest={"run":str(RUN),"source_hashes":baseline,"figure_hashes":{p.name:digest(p) for p in artifacts},"data_rows":81756,"new_statistical_tests":False,"parser_changed":False,"thesis_modified":False,"accuracy_pooling":"descriptive Commonsense + Justice, counts pooled; per-subset tables retained","created_at":datetime.now().isoformat()}
(RUN/"manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print(json.dumps(checks,indent=2))
print("SELESAI:",RUN)
